<a href="https://colab.research.google.com/github/ssprajapati2021/Hybrid-RAG-Fine-Tuning/blob/main/notebook/RAG_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 4: RAG Implementation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] `outputs.json` — Created by Notebook 3
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `./chroma_db/` — Persisted ChromaDB vector index _(Required by NB5 and NB7)_
- [ ] `outputs.json` (updated) — adds `naive_rag_output` _(Required by NB5 and NB7)_

---

In [ ]:
# Mount Google Drive to access the corporate_policies Markdown (.md) files with outputs.json
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
artifact_path = "/content/drive/MyDrive/corporate_policies/";
sop_artifact_path = artifact_path + "sop_documents";

In [ ]:
# Installing Required Packages
#!pip install -q langchain-huggingface sentence-transformers
#!pip install -q langchain-community
#!pip install -q chromadb langchain-chroma
!pip install -q -U bitsandbytes>=0.46.1

In [ ]:
from langchain_chroma import Chroma
print("Chroma imported successfully!")

Chroma imported successfully!


### **Task 3.2: Implement Retrieval-Assisted Generation**

#### **3.2.1 Generate Embeddings [4 marks]**
**The Task:** Initialise the `all-MiniLM-L6-v2` embedding model, embed the SOP documents, and validate the embeddings were produced.

**Hints & Tips:**
* Load the SOP documents with `TextLoader` first (or reuse the corpus from NB2).
* `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")` runs on CPU — no GPU needed.
* Validate by embedding one sample string and checking the vector length (384 dims for MiniLM).
* You MUST use the same embedding model when reloading in Notebooks 5 and 7.

**Embedding Model Options:**
* **`all-MiniLM-L6-v2`** (recommended): 384-dim, fast, ~80MB.
* **`all-mpnet-base-v2`**: 768-dim, higher quality, slower.
* **`bge-small-en-v1.5`**: 384-dim, newer architecture.

**Learner Inference:** Your text is now coordinates in semantic space — similar meanings sit close together.

In [ ]:
import glob
from langchain_community.document_loaders import TextLoader

md_files = glob.glob(f"{sop_artifact_path}/*.md")

print(f"Total SOP files: {len(md_files)}")

documents = []

for file in md_files:
    loader = TextLoader(file, encoding="utf-8")
    documents.extend(loader.load())

print(f"Total Documents Loaded: {len(documents)}")

/tmp/ipykernel_15797/832177068.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Total SOP files: 13
Total Documents Loaded: 13


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Validate the Embedding Model for a single line text
sample_text = "Domestic orders deliver within 3-7 business days."

embedding = embedding_model.embed_query(sample_text)

# Print Vector Sample
print(f"Embedding Dimension: {len(embedding)}")

# Print Vector Sample
print(f"\nVector Sample: {embedding[:10]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Dimension: 384

Vector Sample: [0.012439008802175522, -0.03271061182022095, 0.06439197808504105, -0.004930790513753891, -0.07578543573617935, -0.06563128530979156, -0.1154034435749054, -0.08015567064285278, -0.029033180326223373, 0.0022570942528545856]


In [ ]:
# Generate Embedding for All SOP documents
document_embeddings = embedding_model.embed_documents(
    [doc.page_content for doc in documents]
)

print("Number of Embeddings:", len(document_embeddings))
print("Embedding Dimension:", len(document_embeddings[0]))

Number of Embeddings: 13
Embedding Dimension: 384


#### **3.2.2 Build Vector Index [4 marks]**
**The Task:** Create a persistent Chroma vector index from the embedded documents, configure similarity search, and validate the index.

**Hints & Tips:**
* `Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db")` auto-saves — no manual `.persist()` needed.
* Validate with `vector_db._collection.count()` — should equal the number of SOP documents.
* Run one test `.similarity_search("refund", k=1)` to confirm retrieval works.

**Vector DB Options:**
* **ChromaDB** (recommended): simple API, auto-persistence, LangChain integration.
* **FAISS**: faster for >100K docs, but no built-in persistence (manual serialization).

**Learner Inference:** The index lets you search by meaning — it returns the document mathematically closest to your query's coordinates.

In [ ]:
from langchain_chroma import Chroma

# Define Persistent Directory
persist_directory = f"{artifact_path}/chroma_db"

# Build the Vector Database
vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory=persist_directory
)

# Validate the index
print("Documents in Vector DB:", vector_db._collection.count())

# Test Similarity Search
results = vector_db.similarity_search(
    "refund",
    k=1
)

print(results[0].page_content)

Documents in Vector DB: 26
# Refund Policy

## Eligibility
Customers may request a refund within 30 days of the original purchase or
delivery date, whichever is later. To be eligible, the item must be unused or
defective, and the request must reference a valid order identifier. Digital
goods and subscription fees already consumed are non-refundable except where
required by local consumer law.

## Refund Methods
Approved refunds are issued to the original payment method. If the original
card or wallet is no longer active, store credit of equal value is offered as
an alternative. Cash refunds are not provided for online orders.

## Processing Time
Once a refund is approved, the amount is released within 5–7 business days.
Bank or card-network posting can add a further 3–5 business days, so advise
customers that the full cycle may take up to 12 business days. Duplicate or
accidental charges are prioritized and typically reversed within 3 business
days.

## Partial Refunds
Partial refunds 

#### **3.2.3 Implement Retrieval Workflow [4 marks]**
**The Task:** Execute a "Naive RAG" workflow — pass the raw customer query into the vector DB, fetch the top result, and augment the LLM prompt.

**Hints & Tips:**
* Use `.similarity_search(query, k=1)` for the top-1 document.
* Check whether the raw query retrieved the WRONG policy — common with ambiguous queries.
* Inject context via the system prompt: `"Answer strictly using this SOP: {context}"`.

**Parameter Tuning:**
* `k=1`: one document (focused). `k=3`: more context if SOPs overlap. `k=5`: max, risks long prompts.

**Learner Inference:** Noisy queries often retrieve the wrong document — proving Naive RAG is flawed and motivating the fine-tuned router.

In [ ]:
# Customer query
test_query = "My package is taking much longer than expected. Can you tell me what's happening?"

# Retrieve top-1 document
retrieved_docs = vector_db.similarity_search(
    test_query,
    k=1
)

# Extract retrieved context
context = retrieved_docs[0].page_content

print(context)
print(retrieved_docs[0].metadata)


# Shipping Delays

## Scope
This procedure covers orders that have shipped but are running behind the
estimated delivery date, as well as orders stuck in a pre-shipment state.

## Standard Timelines
Domestic orders deliver within 3–7 business days; international orders within
10–21 business days. An order is only considered "delayed" once it has passed
the upper bound of its quoted window. Carrier scans can lag reality by up to
24 hours, so a single stale tracking event is not by itself a delay.

## Agent Steps
1. Confirm the order identifier and the original estimated window.
2. Check the latest carrier scan and the warehouse dispatch status.
3. If the parcel is in transit but late, open a carrier trace and tell the
   customer to expect an update within 24–48 hours.
4. If the parcel shows no movement for more than 5 business days, treat it as
   potentially lost and offer a replacement shipment or a full refund.
5. For weather, customs, or peak-season backlogs, explain the cause and 

In [ ]:
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig)

# Configure the model which used in notebook2
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load the base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
# Building RAG Prompt

messages = [
    {
        "role": "system",
        "content": f"""Answer strictly using this SOP.
         SOP: {context}
        If the answer is not present in the SOP, say you don't know."""
    },
    {
        "role": "user",
        "content": test_query
    }
]
# Apply chat template
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Tokenize
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

# Generate the RAG Response
outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=False,
    temperature=None,
    top_p=None
)

In [ ]:
# Display only the Assistant Response
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

rag_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(rag_response)

I'm sorry to hear your package is delayed. To better understand the situation, could you please provide the following details:
1. The order identifier (e.g., SKU) and the original estimated delivery date.
2. A summary of any recent updates from the carrier (scan) and the warehouse dispatch status.
Based on these details, I will help you determine if there's a potential issue with the carrier or if the package might be lost. Please share the information so we can address your concern effectively.


In [ ]:
# Verify Retrieval
print("Retrieved Document:")
print(retrieved_docs[0].metadata["source"])

Retrieved Document:
/content/drive/MyDrive/corporate_policies/sop_documents/shipping_delays.md


***Naive RAG successfully retrieved the correct shipping delays policy document.***

---
## Save Artifacts for Downstream Notebooks

In [ ]:
import json

with open(f"{artifact_path}/outputs.json", "r") as f:
    outputs_json = json.load(f)

outputs_json["naive_rag_output"] = rag_response

# Save the updated file
with open(f"{artifact_path}/outputs.json", "w") as f:
    json.dump(outputs_json, f, indent=4)

# Verify
print(json.dumps(outputs_json, indent=4))

{
    "test_query": "My package is taking much longer than expected. Can you tell me what's happening?",
    "ground_truth": "Domestic orders deliver within 3-7 business days.",
    "baseline_output": "I'm sorry to hear that your package is taking longer than expected. There could be several reasons for this delay, such as traffic or road conditions, weather-related delays, or issues with the carrier's system. I recommend checking the status of your package on the carrier's website or by calling their customer service number. They can provide more information and help resolve any issues that may be causing the delay. If the issue persists, it might be best to contact the carrier directly to see if they can expedite delivery.",
    "naive_rag_output": "I'm sorry to hear your package is delayed. To better understand the situation, could you please provide the following details:\n1. The order identifier (e.g., SKU) and the original estimated delivery date.\n2. A summary of any recent upda

---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 5.**

- [x] SOP documents loaded via `TextLoader`
- [x] **Embeddings generated and validated** ← _Task 3.2.1_
- [x] **ChromaDB index built, validated, and persisted** ← _Task 3.2.2_
- [x] Naive similarity search executed on `test_query`
- [x] Naive RAG output generated with SOP context injected
- [x] **`./chroma_db/` exists on disk** ← _CRITICAL for NB5 and NB7_
- [x] **`outputs.json` updated** with `naive_rag_output` ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**